# Phase 2.1 end-to-end workflow

This notebook is the reproducible top-level caller for the independent Phase 2.1 package. It imports the public `Phase21Config`, `run_phase21`, and `save_phase21_result` interfaces, then optionally creates all requested figures.

The default is a dependency and physics smoke run (`RUN_ANALYSIS = False`) so the notebook can be opened without a local TNG snapshot. Set `RUN_ANALYSIS = True`, provide a valid TNG50-1 path, and either provide the eight Henriques cooling tables or select the documented constant-cooling smoke mode before running the full pipeline.

In [1]:
from pathlib import Path
import json
import os
import sys

def locate_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src' / 'analysis' / 'pipeline.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from project/2.1 or its parent directory.')

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT={PROJECT_ROOT}')

PROJECT_ROOT=/public/home/zju_visitor/LiuYuanhao/SAM_project/2.1


In [2]:
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytest

from src import Phase21Config, run_phase21, save_phase21_result
from src.physics.cooling_function import ConstantCoolingFunction, LgalCoolingFunction

print({
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'h5py': h5py.__version__,
    'matplotlib': matplotlib.__version__,
    'pytest': pytest.__version__,
})

# Set RUN_ANALYSIS=True for a real TNG50-1 run.
RUN_ANALYSIS = True
REBUILD_SAMPLE = False
USE_CONSTANT_COOLING = True
CONSTANT_COOLING = 1.0e-23
STATE_WORKERS = int(os.environ.get('PHASE21_STATE_WORKERS', '1'))
STATE_BACKEND = os.environ.get('PHASE21_STATE_BACKEND', 'thread')

BASE_PATH = Path(os.environ.get('TNG50_BASE_PATH', Phase21Config.base_path))
COOLING_TABLE_DIR = PROJECT_ROOT / 'data' / 'external' / 'cooling_tables'
CACHE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'cache'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'results' / 'figures'
config = Phase21Config(
    base_path=str(BASE_PATH),
    state_workers=STATE_WORKERS,
    state_parallel_backend=STATE_BACKEND,
)
print(f'base_path={config.base_path}')
print(f'state_workers={config.state_workers}, backend={config.state_parallel_backend}')
print(f'fingerprint={config.fingerprint}')

{'numpy': '1.26.4', 'pandas': '2.2.2', 'h5py': '3.11.0', 'matplotlib': '3.9.2', 'pytest': '7.4.4'}
base_path=/public/share/chenhouzun/TNG50-1/output
state_workers=1, backend=thread
fingerprint=1a52684b83b6dc46a0672122bb3ec91f56e0f3d481b07b91410eff6ccbe23a39


## Dependency and pure-physics smoke check

This check exercises the installed scientific stack without reading TNG files. It is intentionally independent of the full-data switch above.

In [3]:
from src.physics.feedback import compute_agn_strength
from src.physics.sam_cooling import compute_isothermal_sam_cooling

smoke_cooling = compute_isothermal_sam_cooling(
    m_hot_msun=1.0e10,
    z_hot_mass_fraction=0.01,
    m200c_msun=1.0e12,
    r200c_pkpc=100.0,
    cooling_function=ConstantCoolingFunction(CONSTANT_COOLING),
)
smoke_agn = compute_agn_strength(
    m_hot_msun=1.0e10,
    m_bh_msun=1.0e8,
    r200c_pkpc=100.0,
    v200c_km_s=smoke_cooling['v200c_km_s'],
)
assert np.isfinite(smoke_cooling['rate_sam_isothermal_msun_per_yr'])
assert np.isfinite(smoke_agn['agn_strength'])
print('cooling rate [Msun/yr]:', smoke_cooling['rate_sam_isothermal_msun_per_yr'])
print('A_SAM:', smoke_agn['A_SAM'])

cooling rate [Msun/yr]: 3.3799051849151436
A_SAM: 1.0188325168328771


## Full Phase 2.1 call

The following cell is the notebook equivalent of `scripts/run_analysis.py`: it loads the cooling law, scans the sample and snap90--99 history, classifies snap94 → 95 events including `other`, computes statistics, serializes the result tree, and returns paths for every saved product.

In [5]:
result = None
saved_paths = {}
if RUN_ANALYSIS:
    if USE_CONSTANT_COOLING:
        cooling_function = ConstantCoolingFunction(CONSTANT_COOLING)
    else:
        cooling_function = LgalCoolingFunction.from_directory(COOLING_TABLE_DIR)

    result = run_phase21(
        cooling_function,
        config=config,
        cache_dir=CACHE_DIR,
        rebuild_sample=REBUILD_SAMPLE,
        verbose=True,
    )
    output_prefix = OUTPUT_DIR / 'phase21_snap090_099_seed202608'
    saved_paths = save_phase21_result(result, output_prefix)
    for label, path in saved_paths.items():
        print(f'{label}: {path}')
else:
    print('RUN_ANALYSIS=False: full TNG scan skipped. Set it to True to execute the pipeline.')

[phase21] building/loading snap99 sample
[phase21] building/loading per-halo states
[states] cached 1/543 sub=782373
[states] cached 2/543 sub=778679
[states] cached 3/543 sub=770509
[states] cached 4/543 sub=750117
[states] cached 5/543 sub=759892
[states] cached 6/543 sub=706093
[states] cached 7/543 sub=759617
[states] cached 8/543 sub=733465
[states] cached 9/543 sub=777875
[states] cached 10/543 sub=777083
[states] cached 11/543 sub=736805
[states] cached 12/543 sub=792155
[states] cached 13/543 sub=702925
[states] cached 14/543 sub=728859
[states] cached 15/543 sub=728682
[states] cached 16/543 sub=767312
[states] cached 17/543 sub=746827
[states] cached 18/543 sub=718353
[states] cached 19/543 sub=798586
[states] cached 20/543 sub=782740
[states] cached 21/543 sub=741509
[states] cached 22/543 sub=769884
[states] cached 23/543 sub=739819
[states] cached 24/543 sub=749671
[states] cached 25/543 sub=786934
[states] cached 26/543 sub=766843
[states] cached 27/543 sub=676908
[states

: 

In [ ]:
if result is not None:
    from src.plotting import plot_agn_cooling, plot_composition, plot_feedback_before_accretion

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    plot_agn_cooling(
        result['agn_quartile_statistics'],
        halo_results=result['halo_results'],
        save_path=FIGURE_DIR / 'phase21_agn_quartile_cooling.png',
    )
    plot_composition(
        result['composition_statistics'],
        save_path=FIGURE_DIR / 'phase21_composition.png',
    )
    plot_feedback_before_accretion(
        result['normalized_statistics'],
        save_path=FIGURE_DIR / 'phase21_feedback_before_accretion.png',
    )
    print(f'figures written to {FIGURE_DIR}')
else:
    print('Plotting is deferred until RUN_ANALYSIS=True.')

## Closure and output audit

In [ ]:
if result is not None:
    metadata = result['metadata']
    closure = result['closure_diagnostics']
    composition = result['composition_statistics']
    print('selected halos:', metadata['selected_halo_count'])
    print('fingerprint:', metadata['fingerprint'])
    print('closure:', json.dumps(closure, indent=2))
    print('entry component names:', composition['in_component_names'].tolist())
    print('exit component names:', composition['out_component_names'].tolist())
    assert closure['max_abs_count_in_error'] == 0
    assert closure['max_abs_count_out_error'] == 0
else:
    print('No result to audit; run the full-call cell first.')

## Command-line equivalent

From the `project/2.1` directory, the equivalent full run is:

```bash
python scripts/run_analysis.py --base-path /path/to/TNG50-1/output --cooling-table-dir data/external/cooling_tables --cache-dir data/interim/cache --output-dir data/processed --figure-dir results/figures
```